# 02: Rebuilding BRFSS from CDC's raw annual files

Design decisions
- Input: CDC's SAS transport files (LLCPyyyyXPT.zip, or the unzipped LLCPyyyy.XPT), downloaded in a browser and placed anywhere under datasets/ on Drive (datasets/raw/brfss_annual/ is the intended place). Usable years are the odd ones: even years do not ask about blood pressure, cholesterol or fruit and vegetables; 2023 does not ask about fruit and vegetables. Other files in the folder are ignored.
- Recipe (plan Section 6.5): keep respondents with every selected item present, drop "don't know", "refused" and "missing" codes, recode to the public coding (BMI divided by 100 and rounded). Each year uses its own variable names; a name absent from the file or a raw code outside the expected set stops the notebook.
- Gate G0: the rebuilt 2015 table must equal the public 2015 table (the Kaggle heart table used by task A) in row count and every marginal frequency; row-by-row equality is reported too. The diabetes outcome of task B must equal "Diabetes = 2" on the same rows.
- Outputs on Drive: datasets/derived/brfss/brfss_<year>.parquet with the 21 predictors (19 in 2023), the heart outcome, the state code and the survey weight (never used as predictors). In the repository: presence matrix, record counts per cleaning step, a per-year summary and the harmonised set H for the temporal shift.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyreadstat'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, tempfile, time
from pathlib import Path
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.data import public
try:
    from scax.data import brfss_raw as br
except ImportError:
    raise RuntimeError('The stage 8 source is not loaded. Run the stage 8 batch, then Runtime > Disconnect and delete runtime, then run again.')

seed_everything(42)
import os, re
RAW_DIR = paths.RAW / 'brfss_annual'
OUT_DIR = paths.DERIVED / 'brfss'; OUT_DIR.mkdir(parents=True, exist_ok=True)
# the CDC files are looked for anywhere under the project's datasets folder and its top level,
# zipped (LLCPyyyyXPT.zip) or already unzipped (LLCPyyyy.XPT)
found = []
for root, dirs, names in os.walk(paths.DATASETS):
    dirs[:] = [d for d in dirs if d not in ('derived',)]
    found += [Path(root) / n for n in names]
found += [p for p in paths.DRIVE_ROOT.iterdir() if p.is_file()]
by_year = {}
for f in sorted(found):
    name = f.name.strip().upper()
    if re.search(r'LLCP\d{4}', name) and (name.endswith('.ZIP') or name.endswith('.XPT')):
        y = br.year_of(f)
        if y not in by_year or name.endswith('.ZIP'):
            by_year[y] = f
YEARS = sorted(y for y in by_year if y in br.NAMES)
ignored = sorted(set(by_year) - set(YEARS))
print('found:', {y: str(by_year[y].relative_to(paths.DRIVE_ROOT)) for y in sorted(by_year)})
print('used:', YEARS, '| ignored (not usable):', ignored)
assert 2015 in YEARS, (f'No LLCP2015XPT.zip (or LLCP2015.XPT) anywhere under {paths.DATASETS}. Download it from '
                       'https://www.cdc.gov/brfss/annual_data/2015/files/LLCP2015XPT.zip and put it in ' + str(RAW_DIR))


def process(year):
    t0 = time.time()
    src = by_year[year]
    zipped = src.name.strip().upper().endswith('.ZIP')
    work = Path(tempfile.mkdtemp(dir='/content' if Path('/content').exists() else None))
    xpt = br.extract_xpt(src, work) if zipped else src          # an unzipped file is read in place, never deleted
    try:
        cols = br.xpt_columns(xpt)
        used = br.presence(year, cols)
        need = [n for n in used.values() if n] + [c for c in br.EXTRA if c in cols]
        raw = br.read_xpt(xpt, need)
    finally:
        if zipped:
            xpt.unlink(missing_ok=True)
        work.rmdir()
    table, drops = br.build(raw, year)
    out = OUT_DIR / f'brfss_{year}.parquet'
    table.to_parquet(out, index=False)
    drops.insert(0, 'year', year)
    print(f'{year}: {len(raw):,} raw records, {len(table):,} kept, heart disease {table[br.TARGET].mean():.2%}, '
          f'{time.time() - t0:.0f}s')
    return table, drops, used, out


TABLES, DROPS, USED, FILES_OUT = {}, [], {}, {}

## 2015 and gate G0

In [ ]:
TABLES[2015], d, USED[2015], FILES_OUT[2015] = process(2015)
DROPS.append(d)
X_heart, y_heart = public.LOADERS['brfss2015_heart'](paths.RAW)
G0 = br.compare_with_public(TABLES[2015], X_heart, y_heart)
X_diab, y_diab = public.LOADERS['cdc_diabetes'](paths.RAW)
G0['diabetes_task_outcome_equal'] = bool(len(y_diab) == len(TABLES[2015]) and
                                         np.array_equal((TABLES[2015]['Diabetes'] == 2).astype(int).to_numpy(), y_diab))
print(json.dumps(G0, indent=1))
print(d.to_string(index=False))
if not G0['passed']:
    for c in G0['marginal_mismatches']:
        a = TABLES[2015][c].value_counts().sort_index(); b = (X_heart[c] if c in X_heart else pd.Series(y_heart)).value_counts().sort_index()
        print(c, pd.concat([a.rename('rebuilt'), b.rename('public')], axis=1).fillna(0).astype(int).to_string(), sep='\n')
assert G0['passed'], 'Gate G0 failed: the rebuilt 2015 table differs from the public table (see above)'

## Later years

In [ ]:
for year in [y for y in YEARS if y != 2015]:
    TABLES[year], d, USED[year], FILES_OUT[year] = process(year)
    DROPS.append(d)

In [ ]:
PRES = pd.DataFrame({y: {k: (v or 'not asked') for k, v in USED[y].items()} for y in YEARS}).rename_axis('indicator')
PRES.to_csv(f'{paths.RESULTS}/02_variable_presence.csv')
DROP = pd.concat(DROPS, ignore_index=True)
DROP.to_csv(f'{paths.RESULTS}/02_drop_counts.csv', index=False)
H = [c for c in br.INDICATORS if all(USED[y].get(c) for y in YEARS)]
SUM = []
for y in YEARS:
    t = TABLES[y]
    SUM.append({'year': y, 'records': len(t), 'heart_disease_share': float(t[br.TARGET].mean()),
                'diabetes_share': float((t['Diabetes'] == 2).mean()), 'prediabetes_share': float((t['Diabetes'] == 1).mean()),
                'states': int(t['_STATE'].nunique()) if '_STATE' in t else None, 'predictors': int(sum(c in t for c in br.INDICATORS))})
SUM = pd.DataFrame(SUM)
SUM.to_csv(f'{paths.RESULTS}/02_year_summary.csv', index=False)
print(PRES.to_string()); print(SUM.round(4).to_string(index=False))
print(f'harmonised set H ({len(H)} predictors present in every year used):', H)
write_json(f'{paths.RESULTS}/02_brfss_rebuild.json', {
    'years_used': YEARS, 'years_ignored': ignored, 'gate_G0': G0, 'harmonised_set_H': H,
    'parquet_sha256': {str(y): sha256_file(FILES_OUT[y]) for y in YEARS},
    'raw_file_sha256': {str(y): sha256_file(by_year[y]) for y in YEARS}})
print('written: results/02_variable_presence.csv, 02_drop_counts.csv, 02_year_summary.csv, 02_brfss_rebuild.json')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "02: BRFSS rebuilt from raw CDC files (odd years 2015 to 2023), gate G0 against the public 2015 table, presence matrix, harmonised set"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)